***Import and load structure***

In [1]:
from Bio.PDB import PDBParser
from pathlib import Path

pdb_path = Path("../data/raw/1LYZ.pdb")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1LYZ", pdb_path)

print("Structure loaded:", structure)

Structure loaded: <Structure id=1LYZ>


***count models***

In [2]:
models = list(structure.get_models())
print("Number of models:", len(models))

Number of models: 1


***count chains***

In [3]:
model = models[0]
chains = list(model.get_chains())

print("Chains:")
for chain in chains:
    print(chain.id)

Chains:
A


***count residues***

In [4]:
from Bio.PDB.Polypeptide import is_aa

residue_count = 0

for residue in model.get_residues():
    if is_aa(residue, standard=True):
        residue_count += 1

print("Number of amino acid residues:", residue_count)

Number of amino acid residues: 129


***count atoms***

In [5]:
atom_count = sum(1 for _ in model.get_atoms())
print("Number of atoms:", atom_count)

Number of atoms: 1102


## Structure Summary (1LYZ)

- Models: 1
- Chains: 1 (Chain A)
- Residues: 129 amino acids
- Atoms: 1102

### Interpretation
This confirms that lysozyme is a small, single-chain protein, making it suitable for structure-based protein engineering and computational design workflows.

***Extract sequence***

In [6]:
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

model = structure[0]

sequence = []

for residue in model.get_residues():
    if is_aa(residue, standard=True):
        resname = residue.get_resname().capitalize()

        try:
            sequence.append(protein_letters_3to1[resname])
        except KeyError:
            sequence.append("X")  # unknown residue

sequence_str = "".join(sequence)

print("Sequence length:", len(sequence_str))
print("Sequence (first 50 aa):", sequence_str[:50])

Sequence length: 129
Sequence (first 50 aa): KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGS


***Add full sequence print- for visualization***

In [7]:
print(sequence_str)

KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL


***save sequence to file***

In [8]:
from pathlib import Path

output_path = Path("../data/processed/1LYZ_sequence.txt")

with open(output_path, "w") as f:
    f.write(sequence_str)

print("Saved sequence to:", output_path)

Saved sequence to: ../data/processed/1LYZ_sequence.txt


***verify file***

In [9]:
cat ../data/processed/1LYZ_sequence.txt

KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL

## Sequence–Structure Relationship

The sequence was extracted directly from the structure file rather than using a reference database.

### Why this matters
- ensures consistency with the actual structure used
- accounts for missing or unresolved residues
- avoids mismatch between sequence and structure

### Implication for protein engineering
All future mutations and designs will be based on this sequence, ensuring compatibility with structural constraints.

***creating a structured, reusable data table from Biopython objects***

In [14]:
import pandas as pd
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

model = structure[0]

residue_rows = []

for chain in model.get_chains():
    for residue in chain.get_residues():
        if not is_aa(residue, standard=True):
            continue

        hetflag, resseq, icode = residue.id
        resname_3 = residue.get_resname().capitalize()

        try:
            resname_1 = protein_letters_3to1[resname_3]
        except KeyError:
            resname_1 = "X"

        atom_names = {atom.get_name() for atom in residue}

        has_N = "N" in atom_names
        has_CA = "CA" in atom_names
        has_C = "C" in atom_names
        has_O = "O" in atom_names

        if has_CA:
            ca_coord = residue["CA"].get_coord()
            ca_x, ca_y, ca_z = map(float, ca_coord)
        else:
            ca_x, ca_y, ca_z = None, None, None

        residue_rows.append({
            "chain_id": chain.id,
            "residue_number": resseq,
            "insertion_code": icode.strip() if isinstance(icode, str) else "",
            "residue_name_3": resname_3,
            "residue_name_1": resname_1,
            "has_N": has_N,
            "has_CA": has_CA,
            "has_C": has_C,
            "has_O": has_O,
            "ca_x": ca_x,
            "ca_y": ca_y,
            "ca_z": ca_z,
        })

residue_df = pd.DataFrame(residue_rows)
residue_df.head()

,chain_id,residue_number,insertion_code,residue_name_3,residue_name_1,has_N,has_CA,has_C,has_O,ca_x,ca_y,ca_z
0,A,1,,Lys,K,True,True,True,True,2.439,10.217,9.791
1,A,2,,Val,V,True,True,True,True,2.307,14.172,7.580
2,A,3,,Phe,F,True,True,True,True,-1.187,15.293,7.580
3,A,4,,Gly,G,True,True,True,True,-2.637,17.468,4.864
4,A,5,,Arg,R,True,True,True,True,-3.823,20.764,5.685


***check the total number of residues***

In [18]:
print("Total residues in residue table:", len(residue_df))

Total residues in residue table: 129


***quick summary cell***

In [19]:
print("Total residues in residue table:", len(residue_df))
print()

print("Backbone atom presence:")
print("has_N :", residue_df["has_N"].sum())
print("has_CA:", residue_df["has_CA"].sum())
print("has_C :", residue_df["has_C"].sum())
print("has_O :", residue_df["has_O"].sum())
print()

print("Missing CA coordinates:", residue_df["ca_x"].isna().sum())

Total residues in residue table: 129

Backbone atom presence:
has_N : 129
has_CA: 129
has_C : 129
has_O : 129

Missing CA coordinates: 0


***Save the residue table***

In [20]:
from pathlib import Path

output_path = Path("../data/processed/residue_table.csv")
residue_df.to_csv(output_path, index=False)

print("Saved updated residue table to:", output_path)

Saved updated residue table to: ../data/processed/residue_table.csv


***verify the saved table***

In [24]:
saved_df = pd.read_csv("../data/processed/residue_table.csv")
saved_df.head()

,chain_id,residue_number,insertion_code,residue_name_3,residue_name_1,has_N,has_CA,has_C,has_O,ca_x,ca_y,ca_z
0,A,1,NaN,Lys,K,True,True,True,True,2.439,10.217,9.791
1,A,2,NaN,Val,V,True,True,True,True,2.307,14.172,7.580
2,A,3,NaN,Phe,F,True,True,True,True,-1.187,15.293,7.580
3,A,4,NaN,Gly,G,True,True,True,True,-2.637,17.468,4.864
4,A,5,NaN,Arg,R,True,True,True,True,-3.823,20.764,5.685


## Residue Coordinates and Backbone Completeness

The residue table now includes residue-level structural coordinates using Cα positions and flags indicating whether key backbone atoms are present.

### Why this matters
- Cα coordinates provide a simple residue-level spatial representation
- backbone completeness helps assess structural reliability
- this enables future distance calculations, mutation mapping, and structural filtering

### Current limitation
Cα coordinates are a simplified residue representation. Full side-chain geometry and local environment features will be added later.